# 01 — EDA Overview

**Owner**: Deekshitha (C5)  
**Phase**: 1 — Data Ingestion & EDA

This notebook provides a high-level exploratory analysis of the Pittsburgh EMS & Fire dispatch data.

**Topics covered:**
- Dataset shape, dtypes, null counts
- Service type distribution (EMS vs Fire)
- Priority distribution
- Top call types
- Data completeness distribution
- Year-over-year volume summary

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from config.contracts import COLUMN_MAPPING, RAW_COLUMNS
from src.data.schemas import add_completeness_column, validate_dataframe

pd.set_option('display.max_columns', 20)
print('✅ Imports ready')

## 1. Load Raw Data

In [ ]:
# Load raw CSVs (or Excel if CSVs not available)
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PARENT_DIR = PROJECT_ROOT.parent  # fallback for Excel files

# Try CSV first, fall back to Excel
ems_csv = DATA_DIR / 'EMS_Data.csv'
fire_csv = DATA_DIR / 'Fire_Data.csv'
ems_xlsx = PARENT_DIR / 'EMS_Data.xlsx'
fire_xlsx = PARENT_DIR / 'Fire_Data.xlsx'

if ems_csv.exists():
    print('Loading from CSV...')
    df_ems = pd.read_csv(ems_csv)
    df_fire = pd.read_csv(fire_csv)
elif ems_xlsx.exists():
    print('Loading from Excel (this may take a moment)...')
    df_ems = pd.read_excel(ems_xlsx)
    df_fire = pd.read_excel(fire_xlsx)
else:
    raise FileNotFoundError('No data files found. Run scripts/download_data.py first.')

print(f'EMS records: {len(df_ems):,}')
print(f'Fire records: {len(df_fire):,}')

# Combine into a single DataFrame
df = pd.concat([df_ems, df_fire], ignore_index=True)
print(f'Combined records: {len(df):,}')

## 2. Dataset Overview

In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
print()
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Null counts and percentages
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary = null_summary.sort_values('null_pct', ascending=False)
print('=== Null Summary ===')
null_summary

## 3. Service Type Distribution

In [ ]:
service_counts = df['service'].value_counts()
print(service_counts)

fig = px.pie(
    values=service_counts.values,
    names=service_counts.index,
    title='EMS vs Fire — Call Distribution',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hole=0.4,
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.update_layout(template='plotly_dark', width=600, height=400)
fig.show()

## 4. Priority Distribution

In [ ]:
priority_counts = df['priority'].value_counts().head(15)

fig = px.bar(
    x=priority_counts.index,
    y=priority_counts.values,
    title='Top 15 Priority Codes',
    labels={'x': 'Priority Code', 'y': 'Count'},
    color=priority_counts.values,
    color_continuous_scale='Viridis',
)
fig.update_layout(template='plotly_dark', width=900, height=450)
fig.show()

## 5. Top Call Types

In [ ]:
call_type_counts = df['description_short'].value_counts().head(20)

fig = px.bar(
    x=call_type_counts.values,
    y=call_type_counts.index,
    orientation='h',
    title='Top 20 Call Types (description_short)',
    labels={'x': 'Count', 'y': 'Call Type'},
    color=call_type_counts.values,
    color_continuous_scale='Plasma',
)
fig.update_layout(template='plotly_dark', width=900, height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 6. Data Completeness Distribution

In [ ]:
# Rename columns to NEMSIS-aligned names for completeness scoring
rename_map = {k: v for k, v in COLUMN_MAPPING.items() if k in df.columns}
df_renamed = df.rename(columns=rename_map)

# Add completeness column
df_renamed = add_completeness_column(df_renamed)

print(f'Mean completeness: {df_renamed["data_completeness_pct"].mean():.4f}')
print(f'Median completeness: {df_renamed["data_completeness_pct"].median():.4f}')
print()
print(df_renamed['data_completeness_pct'].describe())

In [ ]:
fig = px.histogram(
    df_renamed,
    x='data_completeness_pct',
    nbins=20,
    title='Data Completeness Distribution (per row)',
    labels={'data_completeness_pct': 'Completeness Score', 'count': 'Records'},
    color_discrete_sequence=['#00CC96'],
)
fig.update_layout(template='plotly_dark', width=800, height=400)
fig.show()

## 7. Year-over-Year Volume

In [ ]:
yearly = df.groupby(['call_year', 'service']).size().reset_index(name='count')

fig = px.bar(
    yearly,
    x='call_year',
    y='count',
    color='service',
    barmode='group',
    title='Year-over-Year Call Volume by Service Type',
    labels={'call_year': 'Year', 'count': 'Call Count', 'service': 'Service'},
    color_discrete_map={'EMS': '#636EFA', 'Fire': '#EF553B'},
)
fig.update_layout(template='plotly_dark', width=900, height=450)
fig.show()

In [ ]:
# Summary statistics
print('=== Year-over-Year Summary ===')
yearly_total = df.groupby('call_year').size()
print(yearly_total)
print(f'\nTotal records: {len(df):,}')
print(f'Unique call types: {df["description_short"].nunique()}')
print(f'Unique cities: {df["city_name"].nunique()}')
print(f'Year range: {df["call_year"].min()} — {df["call_year"].max()}')